### Inference_Predict_Landcover  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import geopandas as gpd
import time
import pickle
import torch
import torch.nn as nn
import torch.optim as optim

import random
from torch.utils.data import * 

In [2]:
if (torch.cuda.is_available()):
    device=torch.device("cuda")
    print('device is:',device)
else:
    device=torch.device("cpu")
    print('device is:',device)

device is: cpu


## Set region name and biome model

In [ ]:
#region_name = 'tambal'        # biome A
#region_name = 'serengeti'     # biome B
#region_name = 'ducos'         # biome C
#region_name = 'machuca'       # biome D
region_name = 'theisoa'       # biome E
#region_name = 'siwa_oasis'    # biome F

In [ ]:
biome_model = 'BiomeE'

## Import AlphaEarth Geotiff from file

In [ ]:
filepath = '../Geotiffs/' + region_name + '_alpha.tif'        # 64 embeddings
filepath

In [ ]:
import rasterio

# Open the GeoTIFF
with rasterio.open(filepath) as src:
    print("Number of bands:", src.count)                    # Number of bands in the file
    print("Height x Width:", src.height, "x", src.width)
    print("CRS:", src.crs)                                  # Coordinate reference system
    print("Transform:", src.transform)                      # Affine transform for georeferencing

    # Read all bands as a 3D numpy array (bands, rows, cols)
    image_alpha = src.read()
    print("Shape of array:", image_alpha.shape)

In [ ]:
image_alpha = image_alpha[:,::-1,:]   # AlphaEarth rows seem to be in reverse order ?
image_alpha.shape

In [ ]:
image_alpha.nbytes/(1024*1024)  

In [ ]:
image_alpha.min(), image_alpha.max()

## Plot AlphaEarth using matplotlib

In [ ]:
img = image_alpha.transpose(1,2,0)               # C x H x W to H x W x C
img = img[:,:,1:4]                               # A01, A02, A03

base = np.min(img)
img = img-base

img = np.minimum(img, 1)
img = np.maximum(img, 0)

img_alpha = img 

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.plot(501,501, 'ro')
plt.title("10 x 10 km AlphaEarth image [A01,A02,A03]")
#plt.savefig("my_plot.jpg", dpi=300, bbox_inches="tight")
plt.show()

## Import Sentinel Geotiff from file

In [ ]:
# sentinel 10-band
filepath = '../Geotiffs/' + region_name + '_sent10.tif'        # 3 bands
filepath

In [ ]:
# sentinel 3-band
filepath = '../Geotiffs/' + region_name + '_sent.tif'        # 3 bands
filepath

In [ ]:
import rasterio

# Open the GeoTIFF
with rasterio.open(filepath) as src:
    print("Number of bands:", src.count)                    # Number of bands in the file
    print("Height x Width:", src.height, "x", src.width)
    print("CRS:", src.crs)                                  # Coordinate reference system
    print("Transform:", src.transform)                      # Affine transform for georeferencing

    # Read all bands as a 3D numpy array (bands, rows, cols)
    image_sent = src.read()
    print("Shape of array:", image_sent.shape)

### Plot Sentinel using matplotlib

In [ ]:
img = image_sent
img = img.transpose(1,2,0)                     # C x H x W to H x W x C
img = img[:,:,0:3]                             # BGR
img = img[:,:,::-1]                            # RGB, reverse the order of the last dimension

print('min =',np.min(img))
print('max =',np.max(img))

img = img/2500

img = np.minimum(img, 1)
img = np.maximum(img, 0)

img_sent = img

plt.figure(figsize=(6,6))
plt.imshow(img)
plt.plot(501,501, 'ro')

plt.title("10 x 10 km image. Sentinel-2 RGB")
#plt.savefig("my_plot1.jpg", dpi=300, bbox_inches="tight")
plt.show()

## Sentinel inference

In [ ]:
# normalise 0 -> 1 

for i in range(10):                                                       # loop through all bands
    p99 = np.percentile(image_sent[i,:,:], 99)
    p98 = np.percentile(image_sent[i,:,:], 98)
    max = np.max(image_sent[i,:,:])
    std = np.std(image_sent[i,:,:])
    #print('channel =',i,' max =', max, ' p99 =', p99, ' p98 =', p98, 'std =', std)
    
image_norm = image_sent/10000 

image_norm = np.minimum(image_norm, 1.0)          # clip values above +1 stds
image_norm = np.maximum(image_norm, 0.0)          # clip values below 0 stds

In [ ]:
print(image_norm.min(), image_norm.max())
print(image_norm.mean(), image_norm.std())

In [ ]:
image_norm.shape, image_sent.shape

### Define pytorch model class - Patch CNN

In [ ]:
# two conv layers and a fully connected layer and a softmax - Song plus FC layer
class Patch(nn.Module):
    def __init__(self):                                               # instantiate in the constructor
        super().__init__()                                            # in - m x 10 x 5 x 5
        self.conv1 = nn.Conv2d(10, 10, kernel_size=3, padding=0)      # out - m x 10 x 3 x 3
        self.act1 = nn.LeakyReLU()
        self.conv2 = nn.Conv2d(10, 20, kernel_size=2, padding=0)      # out - m x 20 x 2 x 2
        self.act2 = nn.LeakyReLU()
        # flatten
        self.fc1 = nn.Linear(80, 40)                                  # out - m x 40
        self.act3 = nn.LeakyReLU()
        self.fc2 = nn.Linear(40, 7)                                   # out logits - m x 7
                                                                      # common practice to output logits without activation function

    def forward(self, x):                            # uses the instances in forward
        out = self.act1(self.conv1(x))
        out = self.act2(self.conv2(out))
        out = out.view(-1, 80)                       # replacing nn.Flatten(), the size -1 is inferred from other dimensions
        out = self.act3(self.fc1(out))
        out = self.fc2(out)
        return out

### Define pytorch model class - Alpha_FC

In [ ]:
# two fully connected layers and a softmax - 64 input features
class Alpha_FC(nn.Module):
    def __init__(self):                                               # instantiate in the constructor
        super().__init__()                                            # in - m x 10
        self.fc1 = nn.Linear(64, 32)                                  # fc1. out - m x 10
        self.act1 = nn.LeakyReLU()                                    # activation 1
        self.fc2 = nn.Linear(32, 16)                                  # fc2. out - m x 10
        self.act2 = nn.LeakyReLU()                                    # activation 2
        self.fc3 = nn.Linear(16,7)                                    # out - m x 6 - 7 classes

    def forward(self, x):                             
        out = self.act1(self.fc1(x))
        out = self.act2(self.fc2(out))
        out = self.fc3(out)       
        return out

## Instantiate a model and load saved weights from file

In [ ]:
model_path = '../Patch_models/patch_' + biome_model + '.pt'
print(model_path)

loaded_model = Patch()           # dont change the model class definition between saving and loading the model
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))

In [ ]:
model_path = '../Patch_models/patch_biomeAll.pt'
print(model_path)

loaded_model = Patch()           # dont change the model class definition between saving and loading the model
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))

In [ ]:
model_path = '../Pixel_models/Alpha_' + biome_model + '.pt'
print(model_path)

loaded_model = Alpha_FC()           # dont change the model class definition between saving and loading the model
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))

In [ ]:
model_path = '../Pixel_models/Alpha_BiomeAll.pt'
print(model_path)

loaded_model = Alpha_FC()           # dont change the model class definition between saving and loading the model
loaded_model.load_state_dict(torch.load(model_path, weights_only=True))

In [ ]:
model=loaded_model.to(device) 

In [ ]:
print(model)

## Run Inference on Sentinel image cube

In [ ]:
## Convert image into Batch format (B x C x H x W)
## due to memory size run inference one row at a time

In [ ]:
p = 5                        # patch size
b = int((p-1)/2)             # boundary for 5x5 patch (p-1)/2
C = image_norm.shape[0]       # number of channels
H = image_norm.shape[1]       # height of cube in pixels
W = image_norm.shape[2]       # width of cube in pixels
C,H,W,p,b

In [ ]:
# run inference on image cube
start = time.time()

model.eval()
index_all = np.zeros([0,(W-2*b)], dtype=int)     # first dummy row of predicted classes

for j in range(b,H-b):                           # all rows
    batch=np.zeros([0,10,5,5])                   # first dummy image
    print('row:',j)
    for i in range(b,W-b):                              # cube column number. b=2, W=1001. range is 2 to 998
        patch = image_norm[:,j-2:j+3,i-2:i+3]           # all channels 5x5 patch [10 x 5 x 5]
        patch = np.expand_dims(patch,0)                 # from [C x H x W] = [1 x C x H x W]
        batch = np.concatenate((batch,patch),axis=0)
    #batch = batch[1:]                                   # remove first blank patch
    
    batch_t = torch.tensor(batch, dtype=torch.float32)  # each batch is one row of patches
    with torch.no_grad():
        batch_t = batch_t.to(device)    
        outputs = model(batch_t)
        pred_index = torch.argmax(outputs, dim=1)       # each output is one row of landcover predictions

    pred_index = np.expand_dims(pred_index,axis=0)               # [997] to [1,997]
    index_all = np.concatenate((index_all,pred_index),axis=0)    # concatenate all rows

#index_all = index_all[1:]    # remove first dummy row

end = time.time()
print('\n\nfinal run time=',(end-start)/60," minutes")

## Run Inference on AlphaEarth image cube

In [ ]:
C = image_alpha.shape[0]       # number of channels
H = image_alpha.shape[1]       # height of cube in pixels
W = image_alpha.shape[2]       # width of cube in pixels
C,H,W

In [ ]:
# run inference on image cube
start = time.time()

model.eval()
index_all = np.zeros([0,W], dtype=int)           # first dummy row of predicted classes

for j in range(0,H):                             # all rows
    batch = image_alpha[:,j,0:W]                 # all channels, row j, all columns
    batch = batch.transpose(1,0)                 # C x W -> W x C
    
    batch_t = torch.tensor(batch, dtype=torch.float32)   # each batch is one row of patches
    with torch.no_grad():
        batch_t = batch_t.to(device)    
        outputs = model(batch_t)
        pred_index = torch.argmax(outputs, dim=1)       # each output is one row of landcover predictions

    pred_index = np.expand_dims(pred_index,axis=0)               # [1000] to [1,1000]
    index_all = np.concatenate((index_all,pred_index),axis=0)    # concatenate all rows

#index_all = index_all[1:]    # remove first dummy row

end = time.time()
print('final run time=',(end-start)/60," minutes")

## Save landcover map to file

In [ ]:
landcover = index_all
landcover.shape

In [ ]:
# Patch CNN Landcover maps
file_name = '../Inference_Maps/' + region_name + '_' + biome_model + '_CNN_Landcover.npy'
file_name

In [ ]:
# Patch CNN Landcover maps
file_name = '../Inference_Maps/' + region_name + '_BiomeAll_CNN_Landcover.npy'
file_name

In [ ]:
# AlphaEarth Landcover maps
file_name = '../Inference_Maps/' + region_name + '_' + biome_model + '_Alpha_Landcover.npy'
file_name

In [ ]:
# AlphaEarth Landcover maps
file_name = '../Inference_Maps/' + region_name + '_BiomeAll_Alpha_Landcover.npy'
file_name

In [ ]:
# save 
np.save(file_name, landcover)      # save landcover image

### Load previously created landcover map

In [ ]:
print(file_name)
load_landcover = np.load(file_name)              # previously created map
load_landcover.shape

## Read Google Earth Screen Grab

In [ ]:
#gge_img = mpimg.imread('../African_Parks/GoogleEarth_Mangochi_zoom1.png')
gge_img = mpimg.imread('../African_Parks/GoogleEarth_Bale_zoom1.png')

In [ ]:
plt.imshow(gge_img)

## Map Names

In [ ]:
Figure_Name = '../Inference_Maps/' + region_name + '_' + biome_model + '_CNN_Landcover.png'
Figure_Name

In [ ]:
Figure_Name = '../Inference_Maps/' + region_name + '_BiomeAll_CNN_Landcover.png'
Figure_Name

In [ ]:
Figure_Name = '../Inference_Maps/' + region_name + '_' + biome_model + '_Alpha_Landcover.png'
Figure_Name

In [ ]:
Figure_Name = '../Inference_Maps/' + region_name + '_BiomeAll_Alpha_Landcover.png'
Figure_Name

### Plot Landcover Map - Only

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

# Define colors for each landcover class
# 0 = Water, 1 = Developed, 2 = Barren, 3 = Trees, 4 = Shrub, 5 = Grass, 6 = Agriculture
class_colors = ["#419BDF", "#C4281B", "#D3CBC4", "#397D49", "#DFC35A", "#88B053", "#FFC0CB"]
cmap = ListedColormap(class_colors)

# Create legend patches
legend_elements = [
    Patch(facecolor=color, label=label)
    for color, label in zip(class_colors, ['Water', 'Developed', 'Barren', 'Trees', 'Shrub', 'Grass', 'Agriculture'])]

fig, ax = plt.subplots(figsize=(6.5, 12))

# whole map
ax.set_xlim(0, 1000)
ax.set_ylim(1000, 0)


ax.imshow(landcover, cmap=cmap, vmin=0, vmax=6)
ax.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_title("Landcover")

scalebar = AnchoredSizeBar(ax.transData,
                           100,               
                           '1 km',            
                           'lower left',      
                           pad=0.5,
                           color='white',
                           frameon=False,
                           size_vertical=2,
                           fontproperties=fm.FontProperties(size=10))
ax.add_artist(scalebar)

# Adjust spacing
plt.tight_layout()
#plt.savefig(Figure_Name, dpi=300, bbox_inches="tight")
plt.show()


## Plot Sentinel Image and Landcover Map side by side

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

# Define colors for each landcover class
# 0 = Water, 1 = Developed, 2 = Barren, 3 = Trees, 4 = Shrub, 5 = Grass, 6 = Agriculture
class_colors = ["#419BDF", "#C4281B", "#D3CBC4", "#397D49", "#DFC35A", "#88B053", "#FFC0CB"]
cmap = ListedColormap(class_colors)

# Create legend patches
legend_elements = [
    Patch(facecolor=color, label=label)
    for color, label in zip(class_colors, ['Water', 'Developed', 'Barren', 'Trees', 'Shrub', 'Grass', 'Agriculture'])]

# Create subplots: 1 row, 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 12))

# whole map
ax1.set_xlim(0, 1000)
ax1.set_ylim(1000, 0)
ax2.set_xlim(0, 1000)
ax2.set_ylim(1000, 0)

ax1.imshow(img_sent)
#ax1.imshow(gge_img)
ax1.set_title("Sentinel-2. True Colour Image")

'''
scalebar = AnchoredSizeBar(ax1.transData,
                           100,               # length of scale bar in data units
                           '1 km',            # label
                           'lower left',      # location
                           pad=0.5,
                           color='white',
                           frameon=False,
                           size_vertical=2,
                           fontproperties=fm.FontProperties(size=10))
ax1.add_artist(scalebar)
'''

ax2.imshow(landcover, cmap=cmap, vmin=0, vmax=6)
ax2.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
ax2.set_title("Landcover Classification")

scalebar = AnchoredSizeBar(ax2.transData,
                           200,               # length of scale bar in data units
                           '2 km',            # label
                           'lower left',      # location
                           pad=0.5,
                           color='white',
                           frameon=False,
                           size_vertical=2,
                           fontproperties=fm.FontProperties(size=10))
ax2.add_artist(scalebar)

# Adjust spacing
plt.tight_layout()
plt.savefig(Figure_Name, dpi=300, bbox_inches="tight")
plt.show()
